In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import glob
import shutil
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
train_path = "/content/drive/MyDrive/tumor/Training"
test_path = "/content/drive/MyDrive/tumor/Testing"

new_train_path = "/content/drive/MyDrive/tumor/Training_3class"
new_test_path = "/content/drive/MyDrive/tumor/Testing_3class"

classes = [
    "meningioma_tumor",
    "no_tumor",
    "pituitary_tumor"
]

In [ ]:
shutil.rmtree(new_train_path, ignore_errors=True)
shutil.rmtree(new_test_path, ignore_errors=True)

os.makedirs(new_train_path)
os.makedirs(new_test_path)

for class_name in classes:
    shutil.copytree(
        os.path.join(train_path, class_name),
        os.path.join(new_train_path, class_name)
    )

    shutil.copytree(
        os.path.join(test_path, class_name),
        os.path.join(new_test_path, class_name)
    )

print("Three-class dataset ready.")

Three-class dataset ready.


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    new_train_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    new_train_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    new_test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

print("Training:", train_ds.class_names)
print("Validation:", val_ds.class_names)
print("Testing:", test_ds.class_names)

Found 2045 files belonging to 3 classes.
Using 1636 files for training.
Found 2045 files belonging to 3 classes.
Using 409 files for validation.
Found 294 files belonging to 3 classes.
Training: ['meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Validation: ['meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Testing: ['meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax")
])


In [ ]:
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop])

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 25s 307ms/step - accuracy: 0.3784 - loss: 1.2688 - val_accuracy: 0.5306 - val_loss: 0.9452
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 121ms/step - accuracy: 0.4927 - loss: 1.0546 - val_accuracy: 0.6504 - val_loss: 0.7967
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.5856 - loss: 0.9103 - val_accuracy: 0.6968 - val_loss: 0.6988
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 114ms/step - accuracy: 0.6400 - loss: 0.8076 - val_accuracy: 0.7677 - val_loss: 0.6235
Epoch 5/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 105ms/step - accuracy: 0.6736 - loss: 0.7289 - val_accuracy: 0.7824 - val_loss: 0.5700
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.7035 - loss: 0.6814 - val_accuracy: 0.8068 - val_loss: 0.5295
Epoch 7/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 7s 130ms/step - accuracy: 0.7249 - loss: 0.6300 - val_accuracy: 0.8166 - val_loss: 0.5055
Epoch 8/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 9s 108ms/step - accuracy: 0.7366 - loss: 0.6121 - val_accuracy: 0

In [ ]:

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)


Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 24s 281ms/step - accuracy: 0.8594 - loss: 0.3595 - val_accuracy: 0.8875 - val_loss: 0.3009
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 94ms/step - accuracy: 0.8998 - loss: 0.2792 - val_accuracy: 0.8802 - val_loss: 0.3140
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 94ms/step - accuracy: 0.9169 - loss: 0.2374 - val_accuracy: 0.9071 - val_loss: 0.2731
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 95ms/step - accuracy: 0.9279 - loss: 0.2062 - val_accuracy: 0.8655 - val_loss: 0.3064
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 93ms/step - accuracy: 0.9432 - loss: 0.1736 - val_accuracy: 0.9218 - val_loss: 0.2194
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.9474 - loss: 0.1477 - val_accuracy: 0.9364 - val_loss: 0.1802
Epoch 7/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - accuracy: 0.9529 - loss: 0.1382 - val_accuracy: 0.9315 - val_loss: 0.1910
Epoch 8/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.9627 - loss: 0.1140 - val_accuracy: 0.9389 -

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Test Accuracy:", accuracy)

10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 318ms/step - accuracy: 0.8980 - loss: 0.2618
Test Accuracy: 0.8979591727256775


In [ ]:

from sklearn.metrics import confusion_matrix, classification_report

y_true = np.concatenate([y.numpy() for x, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
classes = [
    'meningioma_tumor',
    'no_tumor',
    'pituitary_tumor'
]

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=classes))


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step
[[109   5   1]
 [  3 102   0]
 [ 14   7  53]]
                  precision    recall  f1-score   support

meningioma_tumor       0.87      0.95      0.90       115
        no_tumor       0.89      0.97      0.93       105
 pituitary_tumor       0.98      0.72      0.83        74

        accuracy                           0.90       294
       macro avg       0.91      0.88      0.89       294
    weighted avg       0.90      0.90      0.89       294



In [ ]:
model.save("/content/drive/MyDrive/tumor/brain_tumor.keras")